In [20]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  DURABLE EXECUTION & REPLAY MECHANISMS IN AGENTIC SYSTEMS                  ║
# ║  Demo Notebook — Haruto Nakamura $47,000 Brokerage Transfer                ║
# ║                                                                            ║
# ║  Architectural Claim:                                                      ║
# ║  Reliability is a workflow-layer property, not a model-layer property.     ║
# ║  Durable execution — separating deterministic orchestration logic from     ║
# ║  non-deterministic activities and persisting every activity result to a    ║
# ║  durable event history BEFORE returning it to the orchestration layer —    ║
# ║  is the engineering primitive that makes this claim structural.            ║
# ║                                                                            ║
# ╚════════════════════════════════════════════════════════════════════════════╝

In [21]:
# ============================================================================
# CELL 1 — Dependencies
# ============================================================================
# Only external dependency: rich (for colored console output).
# Everything else is Python stdlib.
# Run this once per environment.

!pip install rich

In [23]:
# ============================================================================
# CELL 2 — Imports
# ============================================================================

import sqlite3
import json
import datetime
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich import box

console = Console()

In [24]:
# ============================================================================
# Color Style Constants
# ============================================================================
# Every style uses BOTH foreground and background so text is legible
# on dark terminals, light terminals, and projected slides alike.
#
# Convention:
#   STYLE_OK         → idempotent activity completion (green on dark bg)
#   STYLE_DANGER     → non-idempotent external mutation (white on red bg)
#   STYLE_CHECKPOINT → checkpoint write to event history (white on blue bg)
#   STYLE_REPLAY     → replay skip — activity NOT re-called (white on teal bg)
#   STYLE_CRASH      → crash injection (white on dark red bg)
#   STYLE_WARN       → warning / info after crash (black on yellow bg)
#   STYLE_DIM        → dim supplementary text (grey on near-black bg)

STYLE_OK         = "bold white on dark_green"
STYLE_DANGER     = "bold white on red"
STYLE_CHECKPOINT = "bold white on blue"
STYLE_REPLAY     = "bold white on dark_cyan"
STYLE_CRASH      = "bold white on dark_red"
STYLE_WARN       = "bold black on yellow"
STYLE_DIM        = "white on grey23"
STYLE_SUCCESS    = "bold white on green"

In [25]:
# ============================================================================
# CELL 3 — External Systems Setup
# ============================================================================
# These two SQLite databases simulate external systems that the Haruto
# workflow reaches into. In a production Temporal deployment, the functions
# that call these systems would be wrapped in @activity decorators.
#
# CRITICAL DESIGN DECISION: Both tables are APPEND-ONLY.
# Every call inserts a new row regardless of prior rows.
# This is what makes them non-idempotent — calling them twice
# produces two rows, not one. This mirrors real custodial APIs
# and clearing-system endpoints that do not check for duplicates.

# --- In-memory databases: one for bank holds, one for settlement ---
# Why in-memory? So each notebook run starts clean without leftover files.
# In production these are external services we do NOT control.

bank_db = sqlite3.connect(":memory:")
settlement_db = sqlite3.connect(":memory:")

def init_bank_holds():
    """Create the bank_holds table. Simulates a custodial bank hold API."""
    bank_db.execute("""
        CREATE TABLE IF NOT EXISTS bank_holds (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            workflow_id TEXT,
            customer TEXT,
            amount REAL,
            timestamp TEXT,
            run_number INTEGER
        )
    """)
    bank_db.commit()

def init_settlement_notifications():
    """Create the settlement_notifications table. Simulates downstream clearing system."""
    settlement_db.execute("""
        CREATE TABLE IF NOT EXISTS settlement_notifications (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            workflow_id TEXT,
            recipient TEXT,
            message TEXT,
            hold_id INTEGER,
            timestamp TEXT,
            run_number INTEGER
        )
    """)
    settlement_db.commit()

def reset_bank_holds():
    """Wipe all bank hold records. Used between architecture demos."""
    bank_db.execute("DELETE FROM bank_holds")
    bank_db.commit()

def reset_settlement_notifications():
    """Wipe all settlement notification records. Used between architecture demos."""
    settlement_db.execute("DELETE FROM settlement_notifications")
    settlement_db.commit()

def reset_all():
    """Reset both external systems and return to clean state."""
    reset_bank_holds()
    reset_settlement_notifications()
    console.print("  External systems reset.", style=STYLE_DIM)

# Initialize both tables on first run
init_bank_holds()
init_settlement_notifications()

In [26]:
# ============================================================================
# CELL 4 — Activity Definitions
# ============================================================================
# Each activity simulates one step of the Haruto Nakamura transfer workflow.
# Docstrings document idempotency status and production equivalents.

def verify_account_balance(workflow_id: str, customer: str, amount: float, run_number: int) -> dict:
    """
    Step 1 — Verify account balance.
    IDEMPOTENT: Read-only operation. Calling it twice changes nothing.
    Production equivalent: GET /accounts/{id}/balance
    """
    console.print(f"  ✓ Step 1: Verified {customer}'s balance covers ${amount:,.2f}", style=STYLE_OK)
    return {"step": "verify_balance", "status": "sufficient", "customer": customer, "amount": amount}


def place_bank_hold(workflow_id: str, customer: str, amount: float, run_number: int) -> dict:
    """
    Step 2 — Place bank hold via custodial API.
    ⚠️  NON-IDEMPOTENT: Every call APPENDS a new hold record.
    The bank honors every instruction it receives. Two calls = two holds = $94,000 frozen.
    Production equivalent: POST /holds (no idempotency key)
    """
    ts = datetime.datetime.now().isoformat()
    cursor = bank_db.execute(
        "INSERT INTO bank_holds (workflow_id, customer, amount, timestamp, run_number) VALUES (?, ?, ?, ?, ?)",
        (workflow_id, customer, amount, ts, run_number)
    )
    bank_db.commit()
    hold_id = cursor.lastrowid
    # STYLE_DANGER signals non-idempotent external mutation
    console.print(f"  ⚡ Step 2: Bank hold #{hold_id} placed — ${amount:,.2f} frozen for {customer} (run {run_number})", style=STYLE_DANGER)
    return {"step": "place_hold", "hold_id": hold_id, "amount": amount, "run_number": run_number}


def write_pending_ledger(workflow_id: str, customer: str, amount: float, run_number: int) -> dict:
    """
    Step 3 — Write pending ledger record.
    IDEMPOTENT: Uses upsert logic (in this demo, simply returns same result).
    Production equivalent: PUT /ledger/{workflow_id} (upsert)
    """
    console.print(f"  ✓ Step 3: Pending ledger record written for ${amount:,.2f}", style=STYLE_OK)
    return {"step": "write_ledger", "status": "pending", "amount": amount}


def dispatch_settlement_notification(workflow_id: str, hold_id: int, amount: float, run_number: int) -> dict:
    """
    Step 4 — Dispatch settlement notification to clearing system.
    ⚠️  NON-IDEMPOTENT: Every call APPENDS a new notification record.
    Downstream systems act on every notification they receive.
    Production equivalent: POST /notifications/settlement
    """
    ts = datetime.datetime.now().isoformat()
    message = f"Settlement initiated: ${amount:,.2f} transfer, hold ref #{hold_id}"
    cursor = settlement_db.execute(
        "INSERT INTO settlement_notifications (workflow_id, recipient, message, hold_id, timestamp, run_number) VALUES (?, ?, ?, ?, ?, ?)",
        (workflow_id, "clearing_house", message, hold_id, ts, run_number)
    )
    settlement_db.commit()
    notif_id = cursor.lastrowid
    # STYLE_DANGER signals non-idempotent external mutation
    console.print(f"  ⚡ Step 4: Settlement notification #{notif_id} dispatched to clearing house (run {run_number})", style=STYLE_DANGER)
    return {"step": "dispatch_notification", "notification_id": notif_id, "hold_id": hold_id, "run_number": run_number}


def update_transaction_status(workflow_id: str, run_number: int) -> dict:
    """
    Step 5 — Update transaction status.
    IDEMPOTENT: Sets status to a fixed value. Calling twice yields same state.
    Production equivalent: PUT /transactions/{id}/status
    """
    console.print(f"  ✓ Step 5: Transaction status → 'settling'", style=STYLE_OK)
    return {"step": "update_status", "status": "settling"}


def mark_workflow_complete(workflow_id: str, run_number: int) -> dict:
    """
    Step 6 — Mark workflow complete.
    IDEMPOTENT: Terminal state, re-setting is harmless.
    Production equivalent: PUT /workflows/{id}/status
    """
    console.print(f"  ✓ Step 6: Workflow marked complete", style=STYLE_OK)
    return {"step": "mark_complete", "status": "complete"}

In [27]:
# ============================================================================
# CELL 5 — CrashAt Class & Observer Function
# ============================================================================

class CrashAt:
    """
    Simulates an infrastructure crash at a specific workflow step.
    In production, this is a memory limit, network timeout, or instance preemption —
    events the orchestration layer CANNOT prevent.
    """
    def __init__(self, step: int = None):
        # step=None means no crash (used for the successful second run)
        self.step = step

    def check(self, current_step: int):
        """Raise RuntimeError if current_step matches the crash point."""
        if self.step is not None and current_step == self.step:
            console.print(f"\n  💥 CRASH at step {current_step}!", style=STYLE_CRASH)
            console.print(f"  (Simulates memory limit / network timeout / instance preemption)", style=STYLE_DIM)
            console.print()
            raise RuntimeError(f"Process crashed at step {current_step}")


def show_effects():
    """
    Observer function: queries both external systems and prints
    what the outside world actually experienced.
    This is the ONLY source of truth — process memory is gone after a crash.
    """
    console.print()

    # --- Bank Holds ---
    holds = bank_db.execute("SELECT id, customer, amount, run_number FROM bank_holds ORDER BY id").fetchall()
    t = Table(title="🏦 Bank Holds (External System)", box=box.ROUNDED, title_style="bold white on grey23")
    t.add_column("Hold ID", style="cyan")
    t.add_column("Customer", style="white")
    t.add_column("Amount", style="bold yellow")
    t.add_column("Run #", style="magenta")
    total_frozen = 0.0
    for row in holds:
        t.add_row(str(row[0]), row[1], f"${row[2]:,.2f}", str(row[3]))
        total_frozen += row[2]
    console.print(t)
    console.print(f"  Total frozen: ${total_frozen:,.2f}", style="bold white on grey30")
    console.print()

    # --- Settlement Notifications ---
    notifs = settlement_db.execute("SELECT id, hold_id, message, run_number FROM settlement_notifications ORDER BY id").fetchall()
    t2 = Table(title="📨 Settlement Notifications (External System)", box=box.ROUNDED, title_style="bold white on grey23")
    t2.add_column("Notif ID", style="cyan")
    t2.add_column("Hold Ref", style="white")
    t2.add_column("Message", style="white")
    t2.add_column("Run #", style="magenta")
    for row in notifs:
        t2.add_row(str(row[0]), str(row[1]), row[2], str(row[3]))
    console.print(t2)
    console.print(f"  Total notifications dispatched: {len(notifs)}", style="bold white on grey30")
    console.print()

    return {"total_frozen": total_frozen, "hold_count": len(holds), "notif_count": len(notifs)}

In [28]:
# ============================================================================
# CELL 6 — Architecture A: Stateless Retry
# ============================================================================
# The orchestration logic below is CORRECT.
# Unit tests would pass. Staging would show no issues.
# The failure only emerges when a crash occurs AFTER a non-idempotent
# activity (Step 2: place_bank_hold) has already executed and committed
# to the external system.
#
# Why? Because progress lives in PROCESS MEMORY only.
# When the process dies, all knowledge of completed steps dies with it.
# The retry has no choice but to re-execute everything from Step 1.

WORKFLOW_ID = "haruto-transfer-2024-001"
CUSTOMER = "Haruto Nakamura"
AMOUNT = 47_000.00

def run_stateless_workflow(run_number: int, crash: CrashAt = None):
    """
    Architecture A — Stateless retry workflow.
    Identical function for Run 1 and Run 2. The only difference
    is whether a CrashAt is injected.
    """
    if crash is None:
        crash = CrashAt()  # No crash

    console.print()
    console.print(f"  {'='*56}", style="bold white on grey30")
    console.print(f"  ARCHITECTURE A — Stateless Retry (Run {run_number})", style="bold white on grey30")
    console.print(f"  {'='*56}", style="bold white on grey30")
    console.print()

    # Step 1 — idempotent
    crash.check(1)
    verify_account_balance(WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 2 — ⚠️ NON-IDEMPOTENT: bank hold API
    crash.check(2)
    hold_result = place_bank_hold(WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 3 — idempotent
    crash.check(3)
    write_pending_ledger(WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 4 — ⚠️ NON-IDEMPOTENT: settlement notification
    crash.check(4)
    dispatch_settlement_notification(WORKFLOW_ID, hold_result["hold_id"], AMOUNT, run_number)

    # Step 5 — idempotent
    crash.check(5)
    update_transaction_status(WORKFLOW_ID, run_number)

    # Step 6 — idempotent
    crash.check(6)
    mark_workflow_complete(WORKFLOW_ID, run_number)

    console.print(f"\n  Workflow completed successfully (Run {run_number})", style=STYLE_SUCCESS)
    console.print()

In [29]:
# ============================================================================
# CELL 7 — Run Architecture A (crash_step=4)
# ============================================================================

reset_all()

# --- Run 1: Crash at step 4 ---
# Steps 1-3 execute. Step 2 (non-idempotent) commits a bank hold.
# Crash fires BEFORE step 4 executes, so no notification is sent.
# But the bank hold is already committed to the external system.
try:
    run_stateless_workflow(run_number=1, crash=CrashAt(step=4))
except RuntimeError:
    console.print("  Run 1 terminated. Process memory lost.", style=STYLE_WARN)

# --- Run 2: Retry from scratch (no crash) ---
# The orchestrator has NO RECORD of Run 1's completed steps.
# It re-executes EVERYTHING from Step 1.
# Step 2 fires again → second bank hold → $94,000 frozen.
run_stateless_workflow(run_number=2)

# --- Observe what the external world experienced ---
effects_a = show_effects()

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  OBSERVATION — Architecture A                                          │
# │                                                                        │
# │  Bank hold API calls:              2                                   │
# │  Settlement notifications:         1 (only Run 2 reached Step 4)       │
# │  Total amount frozen:              $94,000.00                          │
# │                                                                        │
# │  WHY: The stateless orchestrator has no memory of Run 1.               │
# │  It re-executed Step 2 (place_bank_hold), which is NON-IDEMPOTENT.    │
# │  The bank honored both instructions because the bank has no way to    │
# │  know the second call was a retry — it looks identical to a new       │
# │  legitimate request.                                                   │
# │                                                                        │
# │  This is SEQUENCING RISK: the crash happened after Step 2 committed   │
# │  but before the workflow finished. The retry duplicated the           │
# │  non-idempotent side effect.                                           │
# │                                                                        │
# │  Haruto now has $94,000 frozen instead of $47,000.                    │
# │  A margin call is reachable. Human intervention is required.           │
# └─────────────────────────────────────────────────────────────────────────┘

console.print(Panel(
    "[bold white on red] SEQUENCING RISK REALIZED [/bold white on red]\n\n"
    f"Bank hold API calls:         [bold]{effects_a['hold_count']}[/bold]\n"
    f"Settlement notifications:    [bold]{effects_a['notif_count']}[/bold]\n"
    f"Total amount frozen:         [bold white on red] ${effects_a['total_frozen']:,.2f} [/bold white on red]\n\n"
    "The stateless orchestrator has no memory of Run 1.\n"
    "It re-executed Step 2 (place_bank_hold), which is NON-IDEMPOTENT.\n"
    "The bank honored both instructions — it cannot distinguish a retry\n"
    "from a new legitimate request.\n\n"
    "Haruto now has $94,000 frozen instead of $47,000.\n"
    "A margin call is reachable. Human intervention is required.",
    title="Observation — Architecture A",
    border_style="red"
))

  External systems reset.

  ========================================================

  ARCHITECTURE A — Stateless Retry (Run 1)

  ========================================================

  ✓ Step 1: Verified Haruto Nakamura's balance covers $47,000.00

...

         🏦 Bank Holds (External System)          
╭─────────┬─────────────────┬────────────┬───────╮
│ Hold ID │ Customer        │ Amount     │ Run # │
├─────────┼─────────────────┼────────────┼───────┤
│ 1       │ Haruto Nakamura │ $47,000.00 │ 1     │
│ 2       │ Haruto Nakamura │ $47,000.00 │ 2     │
╰─────────┴─────────────────┴────────────┴───────╯

  Total frozen: $94,000.00

                     📨 Settlement Notifications (External System)                      
╭──────────┬──────────┬────────────────────────────────────────────────────────┬───────╮
│ Notif ID │ Hold Ref │ Message                                                │ Run # │
├──────────┼──────────┼────────────────────────────────────────────────────────┼───────┤
│ 1        │ 2        │ Settlement initiated: $47,000.00 transfer, hold ref #2 │ 2     │
╰──────────┴──────────┴────────────────────────────────────────────────────────┴───────╯

  Total notifications dispatched: 1

╭───────────────────────────────────────── Observation — Architecture A ──────────────────────────────────────────╮
│  SEQUENCING RISK REALIZED                                                                                       │
│                                                                                                                 │
│ Bank hold API calls:         2                                                                                  │
│ Settlement notifications:    1                                                                                  │
│ Total amount frozen:          $94,000.00                                                                        │
│                                                                                                                 │
│ The stateless orchestrator has no memory of Run 1.                                                              │
│ It re-executed Step 2 (place_bank_hold), which is NON-IDEMPOTENT.                                               │
│ The bank honored both instructions — it cannot distinguish a retry                                              │
│ from a new legitimate request.                                                                                  │
│                                                                                                                 │
│ Haruto now has $94,000 frozen instead of $47,000.                                                               │
│ A margin call is reachable. Human intervention is required.                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [30]:
# ============================================================================
# CELL 8 — Idempotency Audit Checklist
# ============================================================================
#
# Before proceeding to Architecture B, answer these four questions
# about YOUR OWN workflow or system:
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  1. EXTERNAL APIs THAT APPEND STATE                                    │
# │     Does your workflow call any external API where every call creates  │
# │     a new record (holds, charges, orders, reservations)?              │
# │     Your answer: ___________________________________________________  │
# │                                                                        │
# │  2. NOTIFICATIONS OR WEBHOOKS                                          │
# │     Does your workflow dispatch notifications, webhooks, emails, or   │
# │     SMS messages where duplicates would confuse or alarm recipients?  │
# │     Your answer: ___________________________________________________  │
# │                                                                        │
# │  3. FINANCIAL OR LEGAL RECORDS                                         │
# │     Does your workflow write to any ledger, audit trail, or           │
# │     compliance log where duplicate entries have regulatory impact?    │
# │     Your answer: ___________________________________________________  │
# │                                                                        │
# │  4. AUTOMATED DOWNSTREAM PROCESSES                                     │
# │     Does your workflow trigger any downstream automation (settlement, │
# │     fulfillment, deployment) that acts on every signal it receives?   │
# │     Your answer: ___________________________________________________  │
# │                                                                        │
# │  DECISION:                                                             │
# │  If you answered YES to any of the above, your workflow contains      │
# │  non-idempotent activities. A stateless retry architecture exposes    │
# │  you to the same sequencing risk that froze Haruto's $94,000.        │
# │                                                                        │
# │  Document your decision here:                                          │
# │  ________________________________________________________________     │
# │  ________________________________________________________________     │
# │  ________________________________________________________________     │
# └─────────────────────────────────────────────────────────────────────────┘


In [40]:
# ============================================================================
# CELL 9 — Architecture B: Durable Execution with Replay
# ============================================================================

class EventHistory:
    """
    Wraps a SQLite database as a durable event history.
    In production, this role is played by Temporal's event store
    or a Postgres-backed checkpointer.

    The event history is the SINGLE SOURCE OF TRUTH for workflow progress.
    It survives process crashes because it lives OUTSIDE process memory.
    """
    def __init__(self):
        # Why a separate in-memory DB? Simulates a durable store that is
        # independent of the workflow process. In production this would be
        # a networked database, not in-process memory.
        self.db = sqlite3.connect(":memory:")
        self.db.execute("""
            CREATE TABLE IF NOT EXISTS event_log (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                step_name TEXT UNIQUE,
                result_json TEXT,
                timestamp TEXT
            )
        """)
        self.db.commit()

    def record(self, step_name: str, result: dict) -> dict:
        """
        Write the activity result to the event log BEFORE returning it.

        ╔═══════════════════════════════════════════════════════════════╗
        ║  This single write is the entire architectural guarantee.    ║
        ║  Comment it out and Architecture B degrades to               ║
        ║  Architecture A.                                             ║
        ╚═══════════════════════════════════════════════════════════════╝
        """
        ts = datetime.datetime.now().isoformat()
        self.db.execute(
            "INSERT OR REPLACE INTO event_log (step_name, result_json, timestamp) VALUES (?, ?, ?)",
            (step_name, json.dumps(result), ts)
        )
        self.db.commit()
        # STYLE_CHECKPOINT signals a checkpoint write
        console.print(f"  📝 Checkpoint: '{step_name}' recorded to event history", style=STYLE_CHECKPOINT)
        return result

    def get(self, step_name: str):
        """
        Look up a prior result. Returns the recorded dict if found, None otherwise.
        This is the READ side of the replay mechanism.
        """
        row = self.db.execute(
            "SELECT result_json FROM event_log WHERE step_name = ?",
            (step_name,)
        ).fetchone()
        if row:
            return json.loads(row[0])
        return None

    def show(self):
        """Print the event history as a table for pedagogical inspection."""
        rows = self.db.execute("SELECT id, step_name, result_json, timestamp FROM event_log ORDER BY id").fetchall()
        t = Table(title="📋 Event History (Durable Store)", box=box.ROUNDED, title_style="bold white on grey23")
        t.add_column("ID", style="cyan")
        t.add_column("Step", style="bold white")
        t.add_column("Result", style="white", max_width=60)
        t.add_column("Timestamp", style="dim")
        for row in rows:
            t.add_row(str(row[0]), row[1], row[2], row[3])
        console.print(t)


def execute_or_replay(history: EventHistory, step_name: str, activity_fn, *args) -> dict:
    """
    The replay gate. This is the mechanism Temporal implements internally —
    making it explicit is the pedagogical point.

    BEFORE every activity:
      1. Check the event history for a prior result.
      2. If found → return the recorded result WITHOUT calling the activity.
      3. If not found → call the activity, write the result to the history
         BEFORE returning it to the orchestration layer, then return.

    WHY write-before-return? If we returned first and wrote second, a crash
    between return and write would lose the record, and the next replay
    would re-execute the activity — defeating the entire mechanism.
    """
    prior = history.get(step_name)
    if prior is not None:
        # STYLE_REPLAY signals a replay skip — activity is NOT re-executed
        console.print(f"  ↩ Replay: '{step_name}' — returning recorded result (activity NOT called)", style=STYLE_REPLAY)
        return prior

    # No prior result — execute the activity for real
    result = activity_fn(*args)
    # Write to history BEFORE returning to orchestration layer
    history.record(step_name, result)
    return result


def run_durable_workflow(history: EventHistory, run_number: int, crash: CrashAt = None):
    """
    Architecture B — Durable execution with replay.

    IDENTICAL orchestration logic to Architecture A's run_stateless_workflow.
    The ONLY structural difference: every activity passes through
    execute_or_replay(), which checks the event history before calling.

    On Run 2, steps 1-3 replay from history (no external calls).
    Step 4 onwards executes live (first time reaching these steps).
    """
    if crash is None:
        crash = CrashAt()  # No crash

    console.print()
    console.print(f"  {'='*56}", style="bold white on grey30")
    console.print(f"  ARCHITECTURE B — Durable Replay (Run {run_number})", style="bold white on grey30")
    console.print(f"  {'='*56}", style="bold white on grey30")
    console.print()

    # Step 1 — idempotent (replayed on Run 2)
    crash.check(1)
    execute_or_replay(history, "step_1_verify_balance",
        verify_account_balance, WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 2 — ⚠️ NON-IDEMPOTENT (replayed on Run 2 — NOT re-executed)
    crash.check(2)
    hold_result = execute_or_replay(history, "step_2_place_hold",
        place_bank_hold, WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 3 — idempotent (replayed on Run 2)
    crash.check(3)
    execute_or_replay(history, "step_3_write_ledger",
        write_pending_ledger, WORKFLOW_ID, CUSTOMER, AMOUNT, run_number)

    # Step 4 — ⚠️ NON-IDEMPOTENT (executes live on Run 2 — first time)
    crash.check(4)
    execute_or_replay(history, "step_4_dispatch_notification",
        dispatch_settlement_notification, WORKFLOW_ID, hold_result["hold_id"], AMOUNT, run_number)

    # Step 5 — idempotent (executes live on Run 2 — first time)
    crash.check(5)
    execute_or_replay(history, "step_5_update_status",
        update_transaction_status, WORKFLOW_ID, run_number)

    # Step 6 — idempotent (executes live on Run 2 — first time)
    crash.check(6)
    execute_or_replay(history, "step_6_mark_complete",
        mark_workflow_complete, WORKFLOW_ID, run_number)

    console.print(f"\n  Workflow completed successfully (Run {run_number})", style=STYLE_SUCCESS)
    console.print()

In [41]:
# ============================================================================
# CELL 10 — Run Architecture B (crash_step=4)
# ============================================================================

reset_all()

# Create a SINGLE event history that persists across both runs.
# This is the architectural difference — the history survives the crash.
history = EventHistory()

# --- Run 1: Crash at step 4 ---
# Steps 1-3 execute AND are checkpointed to the event history.
# Step 2 (non-idempotent) commits a bank hold AND records the result.
# Crash fires BEFORE step 4 executes.
try:
    run_durable_workflow(history, run_number=1, crash=CrashAt(step=4))
except RuntimeError:
    console.print("  Run 1 terminated. Process memory lost — but event history survives.", style=STYLE_WARN)

console.print()
console.print("  --- Event history after crash ---", style=STYLE_DIM)
history.show()

# --- Run 2: Replay and resume (no crash) ---
# The orchestrator replays from Step 1, but execute_or_replay()
# finds recorded results for steps 1-3 and returns them WITHOUT
# calling the activity functions. Step 2 (place_bank_hold) is
# NEVER handed to the external world a second time.
# Steps 4-6 execute live for the first time.
run_durable_workflow(history, run_number=2)

console.print()
console.print("  --- Event history after successful completion ---", style=STYLE_DIM)
history.show()

# --- Observe what the external world experienced ---
effects_b = show_effects()

# ┌─────────────────────────────────────────────────────────────────────────┐
# │  OBSERVATION — Architecture B                                          │
# │                                                                        │
# │  Bank hold API calls:              1                                   │
# │  Settlement notifications:         1                                   │
# │  Total amount frozen:              $47,000.00                          │
# │                                                                        │
# │  WHY: The durable event history recorded the bank hold result from    │
# │  Run 1. When Run 2 replayed Step 2, execute_or_replay() found the    │
# │  recorded result and returned it WITHOUT calling place_bank_hold().   │
# │  The non-idempotent activity was never handed to the external world   │
# │  a second time.                                                        │
# │                                                                        │
# │  Haruto has exactly $47,000 frozen. No duplication. No margin call.   │
# │  No human intervention required.                                       │
# │                                                                        │
# │  The orchestration logic is IDENTICAL to Architecture A.               │
# │  The reliability difference is entirely structural.                    │
# └─────────────────────────────────────────────────────────────────────────┘

console.print(Panel(
    "[bold white on green] SEQUENCING RISK ELIMINATED [/bold white on green]\n\n"
    f"Bank hold API calls:         [bold]{effects_b['hold_count']}[/bold]\n"
    f"Settlement notifications:    [bold]{effects_b['notif_count']}[/bold]\n"
    f"Total amount frozen:         [bold white on green] ${effects_b['total_frozen']:,.2f} [/bold white on green]\n\n"
    "The durable event history recorded the bank hold result from Run 1.\n"
    "When Run 2 replayed Step 2, execute_or_replay() found the recorded\n"
    "result and returned it WITHOUT calling place_bank_hold().\n"
    "The non-idempotent activity was never re-executed.\n\n"
    "Haruto has exactly $47,000 frozen. No duplication.\n"
    "No margin call. No human intervention required.\n\n"
    "[dim]The orchestration logic is IDENTICAL to Architecture A.\n"
    "The reliability difference is entirely structural.[/dim]",
    title="Observation — Architecture B",
    border_style="green"
))

  External systems reset.

  ========================================================

  ARCHITECTURE B — Durable Replay (Run 1)

  ========================================================

  ✓ Step 1: Verified Haruto Nakamura's balance covers $47,000.00

...

         🏦 Bank Holds (External System)          
╭─────────┬─────────────────┬────────────┬───────╮
│ Hold ID │ Customer        │ Amount     │ Run # │
├─────────┼─────────────────┼────────────┼───────┤
│ 17      │ Haruto Nakamura │ $47,000.00 │ 1     │
╰─────────┴─────────────────┴────────────┴───────╯

  Total frozen: $47,000.00

                      📨 Settlement Notifications (External System)                      
╭──────────┬──────────┬─────────────────────────────────────────────────────────┬───────╮
│ Notif ID │ Hold Ref │ Message                                                 │ Run # │
├──────────┼──────────┼─────────────────────────────────────────────────────────┼───────┤
│ 15       │ 17       │ Settlement initiated: $47,000.00 transfer, hold ref #17 │ 2     │
╰──────────┴──────────┴─────────────────────────────────────────────────────────┴───────╯

  Total notifications dispatched: 1

╭───────────────────────────────────────── Observation — Architecture B ──────────────────────────────────────────╮
│  SEQUENCING RISK ELIMINATED                                                                                     │
│                                                                                                                 │
│ Bank hold API calls:         1                                                                                  │
│ Settlement notifications:    1                                                                                  │
│ Total amount frozen:          $47,000.00                                                                        │
│                                                                                                                 │
│ The durable event history recorded the bank hold result from Run 1.                                             │
│ When Run 2 replayed Step 2, execute_or_replay() found the recorded                                              │
│ result and returned it WITHOUT calling place_bank_hold().                                                       │
│ The non-idempotent activity was never re-executed.                                                              │
│                                                                                                                 │
│ Haruto has exactly $47,000 frozen. No duplication.                                                              │
│ No margin call. No human intervention required.                                                                 │
│                                                                                                                 │
│ The orchestration logic is IDENTICAL to Architecture A.                                                         │
│ The reliability difference is entirely structural.                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [42]:
# ============================================================================
# CELL 11 — Side-by-Side Comparison
# ============================================================================

comparison = Table(title="Architecture A vs Architecture B — Side-by-Side", box=box.DOUBLE_EDGE, title_style="bold white on grey23")
comparison.add_column("Metric", style="bold white", width=32)
comparison.add_column("A — Stateless Retry", style="bold white on red", width=28)
comparison.add_column("B — Durable Replay", style="bold white on dark_green", width=28)

comparison.add_row("Bank hold API calls",         "2",           "1")
comparison.add_row("Settlement notifications",     "1",           "1")
comparison.add_row("Amount frozen on account",     "$94,000.00",  "$47,000.00")
comparison.add_row("Margin call reachable",        "YES",         "NO")
comparison.add_row("Human intervention required",  "YES",         "NO")
comparison.add_row("Where reliability lives",      "Nowhere (hope)", "Workflow layer (event history)")
comparison.add_row("Correctness type",             "Logic-only",  "Logic + durability")

console.print()
console.print(comparison)
console.print()

                         Architecture A vs Architecture B — Side-by-Side                          
╔══════════════════════════════════╤══════════════════════════════╤══════════════════════════════╗
║ Metric                           │ A — Stateless Retry          │ B — Durable Replay           ║
╟──────────────────────────────────┼──────────────────────────────┼──────────────────────────────╢
║ Bank hold API calls              │ 2                            │ 1                            ║
║ Settlement notifications         │ 1                            │ 1                            ║
║ Amount frozen on account         │ $94,000.00                   │ $47,000.00                   ║
║ Margin call reachable            │ YES                          │ NO                           ║
║ Human intervention required      │ YES                          │ NO                           ║
║ Where reliability lives          │ Nowhere (hope)               │ Workflow layer (event        ║
║                                  │                              │ history)                     ║
║ Correctness type                 │ Logic-only                   │ Logic + durability           ║
╚══════════════════════════════════╧══════════════════════════════╧══════════════════════════════╝

In [43]:
# ============================================================================
# CELL 12 — Vary the Crash Step
# ============================================================================
# Architecture B's output is IDENTICAL regardless of crash timing.
# Architecture A's harm profile CHANGES with crash timing:
#   - crash_step=2: crash before bank hold → no harm, but also no progress
#   - crash_step=5: crash after both non-idempotent steps → both duplicate
#
# This demonstrates that Architecture B provides an INVARIANCE GUARANTEE:
# the external world sees the same effects regardless of when the crash occurs.

def run_both_architectures(crash_step: int):
    """Run both architectures with a given crash step and compare."""
    console.print()
    console.print(f"  {'='*66}", style="bold white on grey30")
    console.print(f"  CRASH STEP = {crash_step}", style="bold white on grey30")
    console.print(f"  {'='*66}", style="bold white on grey30")

    # --- Architecture A ---
    reset_all()
    try:
        run_stateless_workflow(run_number=1, crash=CrashAt(step=crash_step))
    except RuntimeError:
        console.print("  A: Run 1 crashed.", style=STYLE_WARN)
    run_stateless_workflow(run_number=2)
    console.print("  Architecture A effects:", style=STYLE_DANGER)
    ea = show_effects()

    # --- Architecture B ---
    reset_all()
    h = EventHistory()
    try:
        run_durable_workflow(h, run_number=1, crash=CrashAt(step=crash_step))
    except RuntimeError:
        console.print("  B: Run 1 crashed.", style=STYLE_WARN)
    run_durable_workflow(h, run_number=2)
    console.print("  Architecture B effects:", style=STYLE_OK)
    eb = show_effects()

    # Summary
    t = Table(title=f"Comparison — crash_step={crash_step}", box=box.ROUNDED, title_style="bold white on grey23")
    t.add_column("Metric", style="bold")
    t.add_column("A", style="bold white on red")
    t.add_column("B", style="bold white on dark_green")
    t.add_row("Bank holds",    str(ea["hold_count"]),  str(eb["hold_count"]))
    t.add_row("Notifications", str(ea["notif_count"]), str(eb["notif_count"]))
    t.add_row("Amount frozen", f"${ea['total_frozen']:,.2f}", f"${eb['total_frozen']:,.2f}")
    console.print(t)
    console.print()


# --- crash_step=2: Crash BEFORE the first non-idempotent activity ---
# Architecture A: Run 1 doesn't reach Step 2, so no harm from Run 1.
#   Run 2 executes everything once. Result: 1 hold, 1 notification, $47,000.
# Architecture B: Same outcome — replay finds nothing, executes all live.
# Both architectures produce correct results when the crash happens
# BEFORE any non-idempotent activity.
run_both_architectures(crash_step=2)

# --- crash_step=5: Crash AFTER both non-idempotent activities ---
# Architecture A: Run 1 executes Steps 1-4 (including both non-idempotent).
#   Run 2 re-executes ALL steps. Result: 2 holds, 2 notifications, $94,000.
# Architecture B: Run 2 replays Steps 1-4 from history. Result: 1 hold, 1 notification, $47,000.
# Architecture A's harm is WORSE here — both non-idempotent steps duplicate.
run_both_architectures(crash_step=5)


  ==================================================================

  CRASH STEP = 2

  ==================================================================

  External systems reset.

  ========================================================

...

         🏦 Bank Holds (External System)          
╭─────────┬─────────────────┬────────────┬───────╮
│ Hold ID │ Customer        │ Amount     │ Run # │
├─────────┼─────────────────┼────────────┼───────┤
│ 22      │ Haruto Nakamura │ $47,000.00 │ 1     │
╰─────────┴─────────────────┴────────────┴───────╯

  Total frozen: $47,000.00

                      📨 Settlement Notifications (External System)                      
╭──────────┬──────────┬─────────────────────────────────────────────────────────┬───────╮
│ Notif ID │ Hold Ref │ Message                                                 │ Run # │
├──────────┼──────────┼─────────────────────────────────────────────────────────┼───────┤
│ 20       │ 22       │ Settlement initiated: $47,000.00 transfer, hold ref #22 │ 1     │
╰──────────┴──────────┴─────────────────────────────────────────────────────────┴───────╯

  Total notifications dispatched: 1

         Comparison — crash_step=5         
╭───────────────┬────────────┬────────────╮
│ Metric        │ A          │ B          │
├───────────────┼────────────┼────────────┤
│ Bank holds    │ 2          │ 1          │
│ Notifications │ 2          │ 1          │
│ Amount frozen │ $94,000.00 │ $47,000.00 │
╰───────────────┴────────────┴────────────╯

In [ ]:
# ============================================================================
# CELL 13 — Exercise: Checkpoint Ablation
# ============================================================================
#
# This exercise lets you break Architecture B and observe degradation.
#
# Instructions:
#
#   1. In Cell 9, find the EventHistory.record() method.
#      Comment out the two lines that write to the database:
#        # self.db.execute(...)
#        # self.db.commit()
#      Leave the console.print line so you can see the "checkpoint" message
#      still fires — but nothing is actually persisted.
#
#   2. Run reset_all() to clear external systems.
#
#   3. Re-run the durable workflow runner from Cell 10:
#        history = EventHistory()
#        try:
#            run_durable_workflow(history, run_number=1, crash=CrashAt(step=4))
#        except RuntimeError:
#            pass
#        run_durable_workflow(history, run_number=2)
#
#   4. Run show_effects() and observe.
#
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  EXPECTED OBSERVATION:                                                  │
# │                                                                        │
# │  Both non-idempotent activities duplicate:                             │
# │    - Bank hold: 2 records → $94,000 frozen (SEQUENCING RISK)          │
# │    - Settlement notification: 1 record (crash was before Step 4,      │
# │      but if crash_step were 5, you'd see 2 — SIDE-EFFECT RISK)       │
# │                                                                        │
# │  Without the checkpoint writes, execute_or_replay() never finds       │
# │  prior results. It calls every activity function on every run.        │
# │  Architecture B has degraded to Architecture A.                        │
# │                                                                        │
# │  This confirms the comment in record():                               │
# │  "This single write is the entire architectural guarantee."            │
# └─────────────────────────────────────────────────────────────────────────┘
#
# OPEN QUESTION:
# What happens when the checkpoint store ITSELF fails? What if the SQLite
# write in record() succeeds for some steps but not others? What if the
# disk fills up, or the network to the durable store times out?
#
# This is the next architectural layer — production systems use Temporal's
# replicated event store or a Postgres-backed checkpointer rather than a
# local SQLite file, precisely because the reliability of the checkpoint
# store is the foundation on which all other reliability guarantees rest.